In [2]:
pip install pandas scikit-learn fairlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 5.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime
from google.colab import drive
drive.mount('/content/drive')

# Modelagem e Pré-processamento
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Métricas de Performance e Justiça
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, f1_score
from fairlearn.metrics import (
    MetricFrame,
    count,
    selection_rate,
    demographic_parity_difference, demographic_parity_ratio,
    equalized_odds_difference, equalized_odds_ratio,
    equal_opportunity_difference, equal_opportunity_ratio
)

# Semente para reprodutibilidade
SEED = 19
# Pasta para salvar os resultados
RESULTS_PATH = "./resultados_bank_churners"
os.makedirs(RESULTS_PATH, exist_ok=True)

def carregar_e_preparar_bank_churners(caminho_csv='BankChurners.csv'):
    caminho_do_arquivo = '/content/drive/MyDrive/BankChurners.csv'
    df = pd.read_csv(caminho_do_arquivo)
    df.head()

    # Remover colunas inúteis e a que causa DATA LEAKAGE
    df = df.iloc[:, :-2]
    df = df.drop(columns=['CLIENTNUM' ], axis=1)

    # Mapear a variável alvo para binário (0 e 1)
    df['Attrition_Flag'] = df['Attrition_Flag'].map({'Existing Customer': 0, 'Attrited Customer': 1})

    # Criar faixas etárias conforme solicitado
    age_bins = [25, 30, 40, 50, 60, 100]
    age_labels = ['25-29', '30-39', '40-49', '50-59', '60+']
    df['Age_Category'] = pd.cut(df['Customer_Age'], bins=age_bins, labels=age_labels, right=False)
    df = df.drop(columns=['Customer_Age'], axis=1)

    # Renomear colunas para corresponder ao padrão do código original, se necessário
    df.rename(columns={'Attrition_Flag': 'target', 'Gender': 'sensitive_sexo'}, inplace=True)
    df['sensitive_sexo'] = df['sensitive_sexo'].map({'M': 1, 'F': 0})


    print(df.info())
    return df

def configurar_preprocessador_bank_churners():
    numeric_features = [
        'Dependent_count', 'Months_on_book', 'Total_Relationship_Count',
        'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit',
        'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
        'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'Total_Revolving_Bal'
    ]

    categorical_features = ['sensitive_sexo', 'Marital_Status', 'Age_Category']

    ordinal_features = ['Education_Level', 'Income_Category', 'Card_Category']

    # Definindo a ordem para cada variável ordinal
    education_order = ['Unknown', 'Uneducated', 'High School', 'College', 'Graduate', 'Post-Graduate', 'Doctorate']
    income_order = ['Unknown', 'Less than $40K', '$40K - $60K', '$60K - $80K', '$80K - $120K', '$120K +']
    card_order = ['Blue', 'Silver', 'Gold', 'Platinum']

    # Criação dos transformadores
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown='ignore', drop='first')
    ordinal_transformer = OrdinalEncoder(categories=[education_order, income_order, card_order])

    # Combinação em um ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features),
            ('ord', ordinal_transformer, ordinal_features)
        ],
        remainder='drop'
    )
    return preprocessor

def calcular_metricas_justica(y_true, y_pred, sensitive_features):
    """Calcula um dicionário com as principais métricas de justiça."""
    fairness_metrics = {
        "demographic_parity_difference": demographic_parity_difference(y_true, y_pred, sensitive_features=sensitive_features),
        "demographic_parity_ratio": demographic_parity_ratio(y_true, y_pred, sensitive_features=sensitive_features),
        "equalized_odds_difference": equalized_odds_difference(y_true, y_pred, sensitive_features=sensitive_features),
        "equal_opportunity_difference": equal_opportunity_difference(y_true, y_pred, sensitive_features=sensitive_features),
    }
    return fairness_metrics


def executar_protocolo():
    # Carregar e preparar o dataset
    dataset = carregar_e_preparar_bank_churners()
    if dataset is None:
        return

    models = {
        'Logistic Regression': {
            'model': LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000, random_state=SEED),
            'params': {'classifier__C': [1.0]}  #Modelo sem penalidade
        },
        'Ridge (L2)': {
            'model': LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000, random_state=SEED),
            'params': {'classifier__C': [0.1, 1, 10]}  # Modelo com penalidade
        },
        'XGBoost': {
            'model': XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=SEED),
            'params': {
                'classifier__learning_rate': [0.1],
                'classifier__max_depth': [5],
                'classifier__n_estimators': [100]
            }
        }
    }

    # Configuração da validação cruzada
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    # Definição das colunas
    target_column = 'target'
    sensitive_col_name = 'sensitive_sexo'
    feature_columns = [col for col in dataset.columns if col not in [target_column]]

    x = dataset[feature_columns]
    y = dataset[target_column]

    # Divisão em treino e teste
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, stratify=y, random_state=SEED)

    # Listas para armazenar todos os resultados
    resultados_performance = []
    resultados_justica = []

    inicio_protocolo = time.time()
    print("\n--- INICIANDO PROTOCOLO DE TREINAMENTO E AVALIAÇÃO ---")

    # Configurar o pré-processador
    preprocessor = configurar_preprocessador_bank_churners()

    # Loop para treinar cada modelo
    for nome_modelo, config in models.items():
        print(f"\nTreinando o modelo: {nome_modelo}... Início: {datetime.now().strftime('%H:%M:%S')}")

        # Criação do pipeline completo
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', config['model'])
        ])

        # GridSearchCV para encontrar os melhores hiperparâmetros
        grid_search = GridSearchCV(pipeline, param_grid=config['params'], cv=cv, scoring='f1_macro', n_jobs=-1)
        grid_search.fit(x_train, y_train)

        best_model_pipeline = grid_search.best_estimator_
        print(f"Melhor F1-Macro (CV): {grid_search.best_score_:.4f}")
        print(f"Melhores parâmetros: {grid_search.best_params_}")

        # Fazer previsões no conjunto de teste
        y_pred_test = best_model_pipeline.predict(x_test)

        # --- Avaliação de Performance ---
        report_dict = classification_report(y_test, y_pred_test, output_dict=True)
        resultados_performance.append({
            "modelo": nome_modelo,
            "f1_macro_cv": grid_search.best_score_,
            "acuracia_teste": accuracy_score(y_test, y_pred_test),
            "recall_teste": recall_score(y_test, y_pred_test),
            "precisao_teste": precision_score(y_test, y_pred_test),
            "f1_teste": f1_score(y_test, y_pred_test),
            "melhores_parametros": grid_search.best_params_
        })

        # --- Avaliação de Justiça ---
        sensitive_features_test = x_test[sensitive_col_name]

        # Usando MetricFrame para métricas por grupo
        metrics_by_group = MetricFrame(
            metrics={'accuracy': accuracy_score, 'selection_rate': selection_rate, 'recall': recall_score},
            y_true=y_test,
            y_pred=y_pred_test,
            sensitive_features=sensitive_features_test
        )

        # Calculando métricas de diferença/razão
        fairness_summary = calcular_metricas_justica(y_test, y_pred_test, sensitive_features_test)

        resultados_justica.append({
            "modelo": nome_modelo,
            **fairness_summary,
            "metricas_por_grupo": metrics_by_group.by_group.to_dict()
        })

    # --- 5. SALVAR RESULTADOS ---
    print("\n--- Salvando resultados ---")

    df_performance = pd.DataFrame(resultados_performance)
    df_justica = pd.DataFrame(resultados_justica)

    df_performance.to_csv(os.path.join(RESULTS_PATH, "resultados_performance.csv"), index=False)
    df_justica.to_csv(os.path.join(RESULTS_PATH, "resultados_justica.csv"), index=False)

    fim_protocolo = time.time()
    print(f"Protocolo concluído em {(fim_protocolo - inicio_protocolo):.2f} segundos.")
    print(f"Resultados salvos na pasta: '{RESULTS_PATH}'")

    # Exibir resultados no console
    print("\n--- Resumo da Performance ---")
    print(df_performance)
    print("\n--- Resumo de Justiça ---")
    print(df_justica.to_string())

# --- Ponto de Entrada do Script ---
if __name__ == "__main__":
    executar_protocolo()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   target                    10127 non-null  int64   
 1   sensitive_sexo            10127 non-null  int64   
 2   Dependent_count           10127 non-null  int64   
 3   Education_Level           10127 non-null  object  
 4   Marital_Status            10127 non-null  object  
 5   Income_Category           10127 non-null  object  
 6   Card_Category             10127 non-null  object  
 7   Months_on_book            10127 non-null  int64   
 8   Total_Relationship_Count  10127 non-null  int64   
 9   Months_Inactive_12_mon    10127 non-null  int64   
 10  Contacts_Count_12_mon     10127 non-null  int64   
 11  Credit_Limit              101

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [15:01:44] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Melhor F1-Macro (CV): 0.9423
Melhores parâmetros: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 5, 'classifier__n_estimators': 100}

--- Salvando resultados ---
Protocolo concluído em 5.08 segundos.
Resultados salvos na pasta: './resultados_bank_churners'

--- Resumo da Performance ---
                modelo  f1_macro_cv  acuracia_teste  recall_teste  \
0  Logistic Regression     0.804415        0.905561      0.584016   
1           Ridge (L2)     0.805614        0.906219      0.581967   
2              XGBoost     0.942317        0.969398      0.870902   

   precisao_teste  f1_teste                                melhores_parametros  
0        0.772358  0.665111                             {'classifier__C': 1.0}  
1        0.778082  0.665885                               {'classifier__C': 1}  
2        0.934066  0.901379  {'classifier__learning_rate': 0.1, 'classifier...  

--- Resumo de Justiça ---
                modelo  demographic_parity_difference  demographic_pari